## Without web search

In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.agents import create_agent


model = init_chat_model(model="gemma4", model_provider="openai", api_key="dummy", base_url="http://localhost:8080/v1")
# model = init_chat_model(model="gemini-3.1-flash-lite", model_provider="google-genai")
agent = create_agent(model=model)

question = HumanMessage(content="Who is the chief minister of Tamil Nadu?")

response = agent.invoke(
    {"messages": [question]}
)

In [5]:
print(response['messages'][-1].content)

I do not have real-time access to the most current political appointments. To get the most accurate and up-to-date information on the Chief Minister of Tamil Nadu, please check a reliable, current news source or the official website of the Tamil Nadu government.


## Add web search tool

In [ ]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for a single, isolated piece of information.
    
    Args:
        query: A concise, atomic search phrase focused on ONE specific entity, 
               person, or metric. NEVER combine multiple questions, use conjunctions 
               like 'and'/'or', or pass full sentences.
    """
    return tavily_client.search(query, max_results=3, include_answer=True)

web_search.invoke("Who is the current CM of tamil nadu?")

{'query': 'Who is the current CM of tamil nadu?',
 'follow_up_questions': None,
 'answer': 'As of 2026, C. Joseph Vijay is the current Chief Minister of Tamil Nadu. He took office on May 10, 2026. His party, Tamilaga Vettri Kazhagam, won the Tamil Nadu Assembly elections that year.',
 'images': [],
 'results': [{'url': 'https://www.jaincollege.ac.in/blogs/list-of-chief-ministers-of-tamil-nadu-1947-2025',
   'title': 'List of Chief Ministers of Tamil Nadu: 1920 – 2026',
   'content': 'M. K. Stalin, son of Karunanidhi, is the current CM of Tamil Nadu. Leading the DMK, he launched the “Illam Thedi Kalvi” (Education at Doorstep) scheme, modernized healthcare with the Makkalai Thedi Maruthuvam initiative, and prioritized digital governance. Stalin represents the new generation of Dravidian leadership, focusing on welfare, urban planning, and technology-driven governance.\n\n### 14. Thiru C. Joseph Vijay (Vijay) (2026– Present) [...] M. G. Ramachandran (MGR): Served as Chief Minister from 19

# Add today date tool

In [14]:
from datetime import datetime
from langchain_core.tools import tool

@tool
def get_current_date() -> str:
    """Returns today's current date and year. 
    
    CRITICAL INSTRUCTION: You MUST call this tool immediately whenever the user asks 
    about current events, things happening "today", "now", "this year", or whenever 
    up-to-date time context is required to ground your search queries or response.
    """
    return datetime.now().strftime("%Y-%m-%d")

In [17]:
agent = create_agent(
    model=model,
    tools=[web_search, get_current_date]
)

question = HumanMessage(content="What are the latest tragedy events that happened in the following locations: 1.Kerala, 2.Nepal, 3.Delhi and 4.Maharashtra?")

response = agent.invoke(
    {"messages": [question]}
)

In [16]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='What are the latest tragedy events that happened in the following locations: 1.Kerala, 2.Nepal, 3.Delhi, 4.Maharashtra and 5.Assam?', additional_kwargs={}, response_metadata={}, id='c2e10b83-3a06-45e2-935f-496c2efb5e54'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 244, 'total_tokens': 255, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 1}}, 'model_provider': 'openai', 'model_name': 'gemma-4-E4B-it-UD-Q4_K_XL.gguf', 'system_fingerprint': 'b9670-02810c7aa', 'id': 'chatcmpl-nYXR2O0oKkrsFulQeQNAbfYpgI5q5PkN', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a07fc3-ecd0-7402-ba21-454755300ebd-0', tool_calls=[{'name': 'get_current_date', 'args': {}, 'id': 'vRk6BX2cB0BiQh6A09W93cFNjzEzxQ7C', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 244, 'output_to

trace: https://smith.langchain.com/public/59432173-0dd6-49e8-9964-b16be6048426/r